# Checkpoint 3 — Feature engineering: temperature and freshness

**Goal:** turn eligible telemetry into three understandable inputs. Run independently with the project `.venv` kernel. No model training yet.

Feature engineering means choosing and calculating useful descriptions of raw data. The model will later learn how those inputs relate to incident risk. We choose definitions now; we do not assign learned risk weights or claim predictive benefit yet.

| Feature | Plain-English question | Unit |
|---|---|---|
| Latest temperature | What is the most recent usable measured temperature? | °C |
| Measurement age | How old is that measurement at the decision time? | Minutes |
| Arrival delay | How long between claimed measurement and receipt? | Minutes |

Age and delay are different. A measurement taken at 9 AM, received at 9:05, and used at 11 AM is **120 minutes old**, with **5 minutes arrival delay**.

We also add a missing-temperature indicator so absence cannot be confused with an actual 0°C reading.


In [1]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
import json

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/events.jsonl").is_file()
             and (p / "src/dispatch_risk/contracts.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the candidate repository.")

def utc(text):
    value = datetime.fromisoformat(text.replace("Z", "+00:00"))
    if value.tzinfo is None:
        raise ValueError("An explicit timezone is required.")
    return value.astimezone(timezone.utc)

with (ROOT / "data/events.jsonl").open() as handle:
    events = [json.loads(line) for line in handle if line.strip()]
versions = {}
for event in events:
    versions.setdefault(event["event_id"], set()).add(event["revision"])
print("Loaded", len(events), "deliveries; no previous notebook is required.")


def known_revisions(records, checkpoint):
    """Select the highest revision received by the checkpoint for each event ID."""
    if checkpoint.tzinfo is None:
        raise ValueError("Checkpoint must include a timezone")
    checkpoint = checkpoint.astimezone(timezone.utc)
    delivered = {}
    latest = {}
    for event in records:
        if utc(event["received_at"]) > checkpoint:
            continue
        key = (event["event_id"], event["revision"])
        # Compare canonical timestamps so equivalent timezone notation agrees.
        normalized = dict(event)
        for field in ("device_time", "received_at"):
            normalized[field] = utc(event[field]).isoformat()
        signature = json.dumps(normalized, sort_keys=True, allow_nan=False)
        if key in delivered:
            if delivered[key] != signature:
                raise ValueError("Conflicting content for the same event/revision")
            continue
        delivered[key] = signature
        prior = latest.get(event["event_id"])
        if prior is not None and prior["shipment_id"] != event["shipment_id"]:
            raise ValueError("An event ID changed shipment")
        if prior is None or event["revision"] > prior["revision"]:
            latest[event["event_id"]] = dict(event)
    return [latest[event_id] for event_id in sorted(latest)]


Loaded 11019 deliveries; no previous notebook is required.


## 1. Define eligibility before calculating features

First apply checkpoint 2's duplicate/revision policy. Then consider temperature records whose measurement time is no later than their receipt time, with a finite numeric value.

**Provisional clock policy for this experiment:** exclude future-clock temperatures where the device time exceeds received time. A real small clock skew can be legitimate; a tolerance or fallback clock is an alternative to investigate. We exclude rather than silently rewrite timestamps. We do not fall back to a superseded revision when its available correction is unusable.

An old but plausible timestamp cannot prove the clock is correct. Features derived from device time inherit this limitation. A reading received after the checkpoint stays unavailable regardless of what its device clock claims.

“Latest” here means the greatest usable device time, not the last row in the file. Ties are broken by received time, then event ID, to make selection repeatable. This helper takes one shipment explicitly. No stale-data cutoff or lookback window is chosen yet: old readings remain present with their age exposed.


In [2]:
import math

def first_features(records, shipment, checkpoint):
    selected = known_revisions(records, checkpoint)
    usable = []
    for event in selected:
        if event["shipment_id"] != shipment or event["kind"] != "temperature_c":
            continue
        value = event["value"]
        if isinstance(value, bool) or not isinstance(value, (float, int)):
            continue
        if not math.isfinite(value):
            continue
        measured = utc(event["device_time"])
        received = utc(event["received_at"])
        if measured > received:  # Exclude measurements whose clocks run ahead of receipt.
            continue
        usable.append(event)
    if not usable:
        return {"latest_temperature_c": None, "measurement_age_minutes": None,
                "arrival_delay_minutes": None, "temperature_missing": 1}
    latest = max(usable, key=lambda e: (utc(e["device_time"]),
                                      utc(e["received_at"]), e["event_id"]))
    measured = utc(latest["device_time"])
    received = utc(latest["received_at"])
    return {
        "latest_temperature_c": float(latest["value"]),
        "measurement_age_minutes": (checkpoint - measured).total_seconds() / 60,
        "arrival_delay_minutes": (received - measured).total_seconds() / 60,
        "temperature_missing": 0,
    }


## 2. Calculate by hand, then check the code

Teaching example: 8°C measured at 9 AM and received at 9:05; corrected to 5°C at noon. At 11 AM we expect **8°C, 120-minute age, 5-minute delay**. At noon the correction still refers to the original 9 AM measurement: age is 180 minutes, and revision arrival delay is 180 minutes.

That last distinction matters: after correction, our delay feature measures the selected revision's availability lag, which can include audit/correction time rather than just sensor transport latency. The feature name needs to reflect that difference.


In [3]:
original = {"event_id": "teaching-reading", "revision": 1,
            "shipment_id": "teaching-shipment", "device_time": "2026-01-01T09:00:00Z",
            "received_at": "2026-01-01T09:05:00Z", "kind": "temperature_c",
            "value": 8.0, "source": "teaching-sensor", "payload": {}}
correction = {**original, "revision": 2, "received_at": "2026-01-01T12:00:00Z", "value": 5.0}
stream = [original, dict(original), correction]
eleven = utc("2026-01-01T11:00:00Z")
features = first_features(stream, "teaching-shipment", eleven)
print("11 AM:", features)
assert features == {"latest_temperature_c": 8.0, "measurement_age_minutes": 120.0,
                    "arrival_delay_minutes": 5.0, "temperature_missing": 0}
print("Noon:", first_features(stream, "teaching-shipment", utc("2026-01-01T12:00:00Z")))
assert first_features(list(reversed(stream)), "teaching-shipment", eleven) == features
assert first_features(stream * 2, "teaching-shipment", eleven) == features
assert first_features([original], "teaching-shipment", eleven) == features
future_clock = {**original, "event_id": "bad-clock", "device_time": "2026-01-01T15:00:00Z"}
assert first_features([future_clock], "teaching-shipment", eleven)["temperature_missing"] == 1
assert first_features([], "teaching-shipment", eleven)["latest_temperature_c"] is None
zero = {**original, "value": 0.0}
assert first_features([zero], "teaching-shipment", eleven)["temperature_missing"] == 0
print("Checks passed: manual calculation, duplicates, ordering, future correction, future clock, missing versus zero.")


11 AM: {'latest_temperature_c': 8.0, 'measurement_age_minutes': 120.0, 'arrival_delay_minutes': 5.0, 'temperature_missing': 0}
Noon: {'latest_temperature_c': 5.0, 'measurement_age_minutes': 180.0, 'arrival_delay_minutes': 180.0, 'temperature_missing': 0}
Checks passed: manual calculation, duplicates, ordering, future correction, future clock, missing versus zero.


## 3. Inspect these features at real decision checkpoints

Select one shipment from the earliest decision checkpoint without hard-coded IDs. This is a descriptive experiment, not a training set or evaluation. Labels are not loaded into the feature helper.


In [4]:
with (ROOT / "data/decision_times.jsonl").open() as handle:
    decisions = [json.loads(line) for line in handle if line.strip()]
first = min(decisions, key=lambda d: (utc(d["decision_time"]), d["shipment_id"]))
shipment = first["shipment_id"]
shipment_records = [e for e in events if e["shipment_id"] == shipment]
checkpoints = sorted((d for d in decisions if d["shipment_id"] == shipment),
                     key=lambda d: utc(d["decision_time"]))
for decision in checkpoints:
    when = utc(decision["decision_time"])
    print(shipment, when.isoformat())
    print(first_features(shipment_records, shipment, when))


s-00000 2026-01-01T08:00:00+00:00
{'latest_temperature_c': 3.798, 'measurement_age_minutes': 60.0, 'arrival_delay_minutes': 35.0, 'temperature_missing': 0}
s-00000 2026-01-01T11:00:00+00:00
{'latest_temperature_c': 5.386, 'measurement_age_minutes': 60.0, 'arrival_delay_minutes': 2.0, 'temperature_missing': 0}
s-00000 2026-01-01T14:00:00+00:00
{'latest_temperature_c': 9.857, 'measurement_age_minutes': 0.0, 'arrival_delay_minutes': 0.0, 'temperature_missing': 0}


## 4. What we decided, and what remains open

- Start with three interpretable measurements and one missing indicator. These establish a calculation we can check by hand; we have not yet measured whether they improve predictions.
- Select by usable measurement time after enforcing received-time availability and revision handling. Selecting by receipt time alone could choose a newly arrived old measurement over a fresher observation.
- Keep missing numeric features as `None`; do not replace missing temperature with 0°C. Model-specific imputation will be fit on training data later.
- No domain temperature threshold, model choice, scaling, alert threshold, or risk probability has been established.
- No lookback window or maximum age has been chosen. A stale reading can still be returned; the model will need missing/staleness behavior and evaluation.
- The helper repeats a small notebook selection function for independence. Final training and serving must share reviewed feature logic in Python modules.

**Interview notes:** “I started with the latest eligible temperature and explicit freshness measurements. I distinguished measurement age from the selected revision's arrival delay and represented missing data separately from zero.”

**Try explaining this:** at 11 AM, a measurement was taken at 10 AM but arrived at 10:50. What are its measurement age and arrival delay? Why might it be useful to keep both?

**Next checkpoint:** trailing temperature summaries and trend. We will explore how a recent window changes the picture and avoid letting duplicate readings distort those summaries. Record progress in [personal/MY_LEARNINGS.md](../MY_LEARNINGS.md).
